In [ ]:
%env CUDA_VISIBLE_DEVICES=7


## 仿真视频数据集

### MoviDVideoDataset
还是刚体仿真  
有质量、摩擦、弹性这些基础物理量；  
这些量会进入仿真；  
但并不是像 PhysGaia / SOPHY 这类那样，针对不同材料显式设置不同 viscosity、Young’s modulus、Poisson’s ratio 等丰富材料参数。


In [ ]:
import tensorflow_datasets as tfds
import torch
from torch.utils.data import Dataset


class MoviDVideoMapDataset(Dataset):
    def __init__(self, builder_dir: str, split: str = "train"):
        super().__init__()
        self.builder = tfds.builder_from_directory(builder_dir)
        self.split = split
        self.info = self.builder.info
        self.num_examples = self.info.splits[split].num_examples

    def __len__(self):
        return self.num_examples

    def __getitem__(self, idx):
        ds = self.builder.as_dataset(split=self.split)
        sample = next(iter(tfds.as_numpy(ds.skip(idx).take(1))))

        video = torch.from_numpy(sample["video"]).permute(0, 3, 1, 2).float() / 255.0
        seg = torch.from_numpy(sample["segmentations"])
        depth = torch.from_numpy(sample["depth"])
        metadata = sample["metadata"]

        return {
            "video": video,
            "segmentations": seg,
            "depth": depth,
            "metadata": metadata,
        }


builder_dir = "/data/gaoya/dataset/kubric_tfds/movi-d/256x256/1.0.0"
dataset = MoviDVideoMapDataset(builder_dir)

sample = dataset[0]
print(sample["video"].shape)
print(sample["video"].max())
print(sample["video"].min())
print(len(dataset))

torch.Size([24, 3, 256, 256])
tensor(0.8392)
tensor(0.)
9750


### PhysGaia
非刚体运动，但是train是螺旋运动相机视角，所以我只下载了test（固定相机视角）只有17个视频

In [ ]:
import os
from pathlib import Path
from typing import Callable, List, Optional, Sequence, Tuple, Union

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset


def _read_txt_paths(
    txt_path: Union[str, Path],
    video_exts: Sequence[str] = (".mp4", ".mov", ".avi", ".webm", ".mkv"),
    strict_exists: bool = True,
) -> List[str]:
    """
    从 txt 中读取视频路径。
    - 一行一个路径
    - 跳过空行
    - 跳过以 # 开头的注释行
    """
    txt_path = Path(txt_path)
    if not txt_path.exists():
        raise FileNotFoundError(f"txt 不存在: {txt_path}")

    video_exts = {x.lower() for x in video_exts}
    paths: List[str] = []

    for line in txt_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        p = Path(line)
        if not p.is_absolute():
            p = (txt_path.parent / p).resolve()
        else:
            p = p.resolve()

        if p.suffix.lower() not in video_exts:
            continue

        if strict_exists and not p.exists():
            print(f"[WARN] 文件不存在，跳过: {p}")
            continue

        paths.append(str(p))

    if len(paths) == 0:
        raise RuntimeError(f"没有从 {txt_path} 中读到有效视频路径")

    return paths


def _sample_frame_indices(total_frames: int, num_samples: Optional[int]) -> np.ndarray:
    """
    均匀采样帧索引。
    - num_samples=None: 取全部帧
    - num_samples>=1: 在 [0, total_frames-1] 中均匀采样
    """
    if total_frames <= 0:
        return np.array([], dtype=np.int64)

    if num_samples is None or num_samples >= total_frames:
        return np.arange(total_frames, dtype=np.int64)

    return np.linspace(0, total_frames - 1, num_samples, dtype=np.int64)


def _read_video_cv2(
    video_path: str,
    num_frames: Optional[int] = None,
    resize: Optional[Tuple[int, int]] = None,
) -> Tuple[np.ndarray, float, int]:
    """
    用 OpenCV 读取视频。
    返回:
        frames: [T, H, W, 3], uint8, RGB
        fps: float
        total_frames: 原视频总帧数
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"无法打开视频: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 0:
        fps = 0.0

    frame_indices = _sample_frame_indices(total_frames, num_frames)
    if len(frame_indices) == 0:
        cap.release()
        raise RuntimeError(f"视频没有可读取帧: {video_path}")

    frames = []
    current_target_ptr = 0
    target_set = set(frame_indices.tolist())

    frame_id = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if frame_id in target_set:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            if resize is not None:
                width, height = resize
                frame_rgb = cv2.resize(frame_rgb, (width, height), interpolation=cv2.INTER_LINEAR)
            frames.append(frame_rgb)

            current_target_ptr += 1
            if current_target_ptr >= len(frame_indices):
                break

        frame_id += 1

    cap.release()

    if len(frames) == 0:
        raise RuntimeError(f"没有成功读取任何帧: {video_path}")

    frames = np.stack(frames, axis=0)  # [T, H, W, 3]
    return frames, fps, total_frames


class PhysGaiaTxtVideoDataset(Dataset):
    """
    从 txt 中读取 mp4 路径，建立 torch Dataset。

    返回 sample 格式:
    {
        "video": Tensor[T, C, H, W], float32, 默认范围 [0, 1]
        "path": str,
        "num_frames": int,         # 原视频总帧数
        "sampled_frames": int,     # 实际取出的帧数
        "fps": float,
    }
    """

    def __init__(
        self,
        txt_path: Union[str, Path],
        num_frames: Optional[int] = None,
        resize: Optional[Tuple[int, int]] = None,
        to_float: bool = True,
        normalize_01: bool = True,
        transform: Optional[Callable] = None,
        strict_exists: bool = True,
    ):
        """
        Args:
            txt_path: txt 文件路径，一行一个视频路径
            num_frames: 每个视频均匀采样多少帧；None 表示全取
            resize: (width, height)，例如 (256, 256)；None 表示不缩放
            to_float: 是否转成 float32
            normalize_01: 是否除以 255 归一化到 [0, 1]
            transform: 对 video tensor 的额外变换，输入输出都应是 Tensor[T,C,H,W]
            strict_exists: True 时跳过不存在的路径；False 时允许保留，getitem 再报错
        """
        self.txt_path = str(txt_path)
        self.video_paths = _read_txt_paths(txt_path, strict_exists=strict_exists)
        self.num_frames = num_frames
        self.resize = resize
        self.to_float = to_float
        self.normalize_01 = normalize_01
        self.transform = transform

    def __len__(self) -> int:
        return len(self.video_paths)

    def __getitem__(self, idx: int):
        video_path = self.video_paths[idx]

        frames, fps, total_frames = _read_video_cv2(
            video_path=video_path,
            num_frames=self.num_frames,
            resize=self.resize,
        )
        # frames: [T, H, W, 3], uint8

        video = torch.from_numpy(frames).permute(0, 3, 1, 2)  # [T, C, H, W]

        if self.to_float:
            video = video.float()
            if self.normalize_01:
                video = video / 255.0

        if self.transform is not None:
            video = self.transform(video)

        sample = {
            "video": video,                     # [T, C, H, W]
            "path": video_path,
            "num_frames": total_frames,         # 原视频总帧数
            "sampled_frames": video.shape[0],   # 实际采样帧数
            "fps": fps,
        }
        return sample
txt_path = "/home/gaoya/Code_Video/Code_data/txts/PhysGaia_testmp4.txt"

dataset = PhysGaiaTxtVideoDataset(
    txt_path=txt_path,
    num_frames=24,          # 每个视频均匀采样 24 帧
    resize=(256, 256),      # resize 到 256x256
)

print("dataset size =", len(dataset))

sample = dataset[0]
print(sample["path"])
print(sample["video"].shape)      # [T, C, H, W]
print(sample["video"].max())
print(sample["video"].min())
print(sample["num_frames"])
print(sample["sampled_frames"])
print(sample["fps"])

dataset size = 36
/data/gaoya/dataset/mijeongkim-PhysGaia/131_data/intern/gunhee/PhysGaia_multi/release_new_2/gas_box_smoke/videos_test/gas_box_smoke_cam1.mp4
torch.Size([24, 3, 256, 256])
tensor(1.)
tensor(0.)
240
24
24.0


## Simulation-Ready的3D asset数据集

### physx-3d


In [2]:
import pathlib
from pathlib import Path
from physxnet_articulation_demo import prepare_physxnet_object
prepared = prepare_physxnet_object(
    physx_root=Path("/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet"),
    version="version_1",
    object_id="39264",
    output_root=Path("/home/gaoya/Code_Video/Code_data/vis"),
    voxel_pitch=0.025,
    json_override= None,
    object_scale_mult=1,
)

/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/trimesh/triangles.py:302: RuntimeWarning: divide by zero encountered in divide
  center_mass = integrated[1:4] / volume
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/trimesh/triangles.py:316: RuntimeWarning: invalid value encountered in scalar multiply
  integrated[5] + integrated[6] - (volume * (center_mass[[1, 2]] ** 2).sum())
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/trimesh/triangles.py:319: RuntimeWarning: invalid value encountered in scalar multiply
  integrated[4] + integrated[6] - (volume * (center_mass[[0, 2]] ** 2).sum())
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/trimesh/triangles.py:322: RuntimeWarning: invalid value encountered in scalar multiply
  integrated[4] + integrated[5] - (volume * (center_mass[[0, 1]] ** 2).sum())
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/trimesh/triangles.py:324: RuntimeWarning: invalid value encountered in scalar mu

In [4]:
from typing import Any, Dict, List, Optional, Tuple

import json
json_path = "/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/finaljson/39264.json"
objs_dir = "/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/partseg/39264/objs"

with open(json_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
def parse_group_info(group_info: Dict[str, Any]) -> Dict[str, Any]:
    parsed: Dict[str, Any] = {"base_group": group_info.get("0", [])}
    movable_groups: List[GroupRecord] = []
    for key, value in group_info.items():
        if key == "0":
            continue
        child_labels = list(value[0]) if isinstance(value[0], list) else [int(value[0])]
        parent_group = str(value[1])
        params = convert_joint_params_yup_to_zup(list(value[2]))
        joint_type = str(value[3])
        movable_groups.append(
            GroupRecord(
                group_id=str(key),
                child_labels=[int(x) for x in child_labels],
                parent_group=parent_group,
                params=params,
                joint_type=joint_type,
            )
        )
    parsed["movable_groups"] = movable_groups
    return parsed
parsed_groups = parse_group_info(meta.get("group_info", {}))
parsed_groups

{'base_group': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
 'movable_groups': []}

In [11]:
group_info.items()

dict_items([('0', [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])])

In [ ]:
group_info = meta.get("group_info", {})
parsed: Dict[str, Any] = {"base_group": group_info.get("0", [])}
movable_groups: List[GroupRecord] = []
for key, value in group_info.items():
    if key == "0":
        continue
    child_labels = list(value[0]) if isinstance(value[0], list) else [int(value[0])]
    parent_group = str(value[1])
    params = convert_joint_params_yup_to_zup(list(value[2]))
    joint_type = str(value[3])
    movable_groups.append(
        GroupRecord(
            group_id=str(key),
            child_labels=[int(x) for x in child_labels],
            parent_group=parent_group,
            params=params,
            joint_type=joint_type,
        )
    )
parsed["movable_groups"] = movable_groups

In [3]:
import genesis as gs

########################## init ##########################
# gs.init()

########################## create a scene ##########################

scene = gs.Scene(
    sim_options=gs.options.SimOptions(
        dt       = 4e-3,
        substeps = 10,
    ),
    mpm_options=gs.options.MPMOptions(
        lower_bound   = (-0.5, -1.0, 0.0),
        upper_bound   = (0.5, 1.0, 1),
    ),
    vis_options=gs.options.VisOptions(
        visualize_mpm_boundary = True,
    ),
    viewer_options=gs.options.ViewerOptions(
        camera_fov=30,
    ),

)

########################## entities ##########################
plane = scene.add_entity(
    morph=gs.morphs.Plane(),
)

obj_elastic = scene.add_entity(
    material=gs.materials.MPM.Elastic(),
    morph=gs.morphs.Box(
        pos  = (0.0, -0.5, 0.25),
        size = (0.2, 0.2, 0.2),
    ),
    surface=gs.surfaces.Default(
        color    = (1.0, 0.4, 0.4),
        vis_mode = 'visual',
    ),
)

obj_sand = scene.add_entity(
    material=gs.materials.MPM.Liquid(),
    morph=gs.morphs.Box(
        pos  = (0.0, 0.0, 0.25),
        size = (0.3, 0.3, 0.3),
    ),
    surface=gs.surfaces.Default(
        color    = (0.3, 0.3, 1.0),
        vis_mode = 'particle',
    ),
)

obj_plastic = scene.add_entity(
    material=gs.materials.MPM.ElastoPlastic(),
    morph=gs.morphs.Sphere(
        pos  = (0.0, 0.5, 0.35),
        radius = 0.1,
    ),
    surface=gs.surfaces.Default(
        color    = (0.4, 1.0, 0.4),
        vis_mode = 'particle',
    ),
)


########################## build ##########################
scene.build()

horizon = 1000
for i in range(horizon):
    scene.step()

[Genesis] [13:23:30] [WARNING] Using a simulation timestep smaller than 2ms is not recommended for 'use_gjk_collision=False' as it could lead to numerically unstable collision detection.
[Genesis] [13:23:30] [INFO] Scene <6603634> created.
[Genesis] [13:23:30] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <6c9bee5>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [13:23:30] [INFO] Adding <gs.MPMEntity>. idx: 1, uid: <42ee0e6>, morph: <gs.morphs.Box>, material: <gs.materials.MPM.Elastic>.
[Genesis] [13:23:30] [INFO] Sampling particles with pbs-32 sampler and generating `.ptc` file:
[Genesis] [13:23:37] [INFO] Sampled 8,182 particles.
[Genesis] [13:23:37] [INFO] Adding <gs.MPMEntity>. idx: 2, uid: <d10780e>, morph: <gs.morphs.Box>, material: <gs.materials.MPM.Liquid>.
[Genesis] [13:23:37] [INFO] Sampling particles with pbs-32 sampler and generating `.ptc` file:
[Genesis] [13:23:38] [INFO] Sampled 27,221 particles.
[Genesis] [13:23:38] [INFO] Adding <gs.MPMEntity>. idx:

/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/quadrants/_test_tools/warnings_helper.py:11: UserWarning: cannot create weak reference to 'tuple' object. Template mapper caching disabled.
  warnings.warn(message)


[Genesis] [13:23:51] [INFO] Building visualizer...
[Genesis] [13:23:58] [INFO] Running at 90.62 FPS.
[Genesis] [13:23:58] [INFO] Running at 90.65 FPS.
[Genesis] [13:23:58] [INFO] Running at 90.75 FPS.
[Genesis] [13:23:58] [INFO] Running at 90.85 FPS.
[Genesis] [13:23:58] [INFO] Running at 90.96 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.06 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.14 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.18 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.20 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.25 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.29 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.31 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.37 FPS.
[Genesis] [13:23:58] [INFO] Running at 91.42 FPS.
[Genesis] [13:23:59] [INFO] Running at 91.46 FPS.
[Genesis] [13:23:59] [INFO] Running at 91.44 FPS.
[Genesis] [13:23:59] [INFO] Running at 91.45 FPS.
[Genesis] [13:23:59] [INFO] Running at 91.46 FPS.
[Genesis] [13:23:59] [INFO] Running at 91.46 FPS.